# 🌾 Division 1 — Data Cleaning & Exploratory Analysis
**Pipeline:** AI Precision Fertilizer System (TNAU-STCR Standard)

**What happens here:**
1. Load `Fertilizer_Recommendation_Dataset.csv`
2. Handle nulls, duplicates, outliers
3. Encode categoricals (LabelEncoder)
4. EDA plots
5. Save `df_clean.pkl` + `label_encoders.pkl` for Division 2

In [ ]:
# Cell 1 | Install dependencies (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Cell 2 | Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('Libraries loaded')

In [ ]:
# Cell 3 | Load Dataset
DATA_PATH = Path(r'D:\project sem 7\my-finalyear-project\UV\fertilizer\dats\Fertilizer_Recommendation_Dataset.csv')
OUT_DIR = DATA_PATH.parent  # dats/ folder

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Cell 4 | Basic Info
print('=== DTYPES ===')
df.info()
print()
df.describe().T

In [ ]:
# Cell 5 | Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
report = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
print('=== MISSING VALUE REPORT ===')
print(report[report.Missing > 0])

# Drop rows where core soil/yield are NaN
core = ['Nitrogen_mg_kg','Phosphorus_mg_kg','Potassium_mg_kg','Soil_pH','Target_Yield_ton_per_Hectare']
df.dropna(subset=core, inplace=True)

# Fill remaining numeric NaN with column median
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
print(f'Rows after cleaning: {len(df)}')

In [ ]:
# Cell 6 | Duplicates
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
df.drop_duplicates(inplace=True)
print(f'Rows after dedup: {len(df)}')

In [ ]:
# Cell 7 | Categorical Inventory
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} unique -> {list(df[col].unique())}')

In [ ]:
# Cell 8 | Encode Categoricals
# Keep raw + encoded for interpretability
le_map = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    le_map[col] = le
print('Encoded columns:', [c + '_enc' for c in cat_cols])

In [ ]:
# Cell 9 | Outlier Clipping (0.5-99.5 percentile)
# Agronomic extremes are real sensor data, clip rather than drop
npk_cols = ['Nitrogen_mg_kg','Phosphorus_mg_kg','Potassium_mg_kg']
for col in npk_cols:
    lo = df[col].quantile(0.005)
    hi = df[col].quantile(0.995)
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lo, hi)
    print(f'{col}: {n_out} extreme values clipped to [{lo:.1f}, {hi:.1f}]')

## 📊 EDA Visualisations

In [ ]:
# Cell 10 | Crop Distribution + Fertilizer Label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

crop_counts = df['Crop_Type'].value_counts()
axes[0].bar(crop_counts.index, crop_counts.values, color='#4CAF50')
axes[0].set_title('Samples per Crop Type', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

fert_counts = df['Fertilizer_Recommendation'].value_counts()
axes[1].pie(fert_counts.values, labels=fert_counts.index,
            autopct='%1.1f%%', colors=['#FF7043','#42A5F5','#66BB6A','#AB47BC'])
axes[1].set_title('Fertilizer Label Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_crop_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11 | NPK Boxplot by Crop
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, nut in zip(axes, npk_cols):
    df.boxplot(column=nut, by='Crop_Type', ax=ax)
    ax.set_title(nut, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('NPK Soil Levels by Crop (mg/kg)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_npk_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12 | Correlation Heatmap
heatmap_cols = [
    'Nitrogen_mg_kg','Phosphorus_mg_kg','Potassium_mg_kg',
    'Soil_pH','Soil_Moisture_Percent','Soil_Organic_Matter_Percent',
    'Soil_Temperature_C','Air_Temperature_C','Rainfall_mm',
    'Target_Yield_ton_per_Hectare'
]
fig, ax = plt.subplots(figsize=(11, 8))
corr = df[heatmap_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 13 | Target Yield Distribution
fig, ax = plt.subplots(figsize=(10, 4))
col_name = 'Target_Yield_ton_per_Hectare'
df[col_name].hist(bins=40, color='#26C6DA', edgecolor='white', ax=ax)
mean_val = df[col_name].mean()
ax.axvline(mean_val, color='red', linestyle='--',
           label=f'Mean: {mean_val:.2f} t/ha')
ax.set_title('Target Yield Distribution', fontweight='bold')
ax.set_xlabel('Target Yield (ton/ha)')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_yield_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Save Clean Data

In [ ]:
# Cell 14 | Dataset Summary
print('=== FINAL CLEAN DATASET ===')
print(f'Total rows     : {len(df):,}')
print(f'Total columns  : {df.shape[1]}')
print(f'Crop types     : {sorted(df["Crop_Type"].unique())}')
print(f'Soil types     : {sorted(df["Soil_Type"].unique())}')
print(f'Growth stages  : {sorted(df["Growth_Stage"].unique())}')
print(f'Irrigation     : {sorted(df["Irrigation_Type"].unique())}')
print()
print('NPK ranges (mg/kg):')
for c in npk_cols:
    print(f'  {c}: [{df[c].min():.1f}, {df[c].max():.1f}], mean={df[c].mean():.1f}')

In [ ]:
# Cell 15 | Save Outputs
df.to_pickle(OUT_DIR / 'df_clean.pkl')
df.to_csv(OUT_DIR / 'df_clean.csv', index=False)

with open(OUT_DIR / 'label_encoders.pkl', 'wb') as f:
    pickle.dump(le_map, f)

print('Saved:')
print(f'  {OUT_DIR / "df_clean.pkl"}')
print(f'  {OUT_DIR / "df_clean.csv"}')
print(f'  {OUT_DIR / "label_encoders.pkl"}')
print(f'\nShape: {df.shape}')
print('\nReady for Division 2 -> 02_stcr_engine.ipynb')